# 09 — GraphRAG and Network Knowledge Graphs

**Network LLM Engineering — Part II — Knowledge and Context**

### Learning goals
- Understand why networks are naturally graphs
- Build simple topology-aware retrieval
- Differentiate GraphRAG from ordinary vector RAG

In [ ]:
%pip install -q networkx

## Why GraphRAG is especially relevant to networking

Networks are already graphs:
- nodes: devices, interfaces, VRFs, prefixes, applications,
- edges: connected_to, peers_with, advertises, depends_on, hosted_on.

Vector RAG answers "which text chunks are semantically similar?"
Graph retrieval can answer "what is connected to this failed interface?" or "which applications depend on this firewall?"

In [ ]:
import networkx as nx

G = nx.DiGraph()
edges = [
    ("Client-VLAN120","Leaf1","gateway_on"),
    ("Leaf1","Spine1","connected_to"),
    ("Leaf1","Spine2","connected_to"),
    ("Spine1","Leaf2","connected_to"),
    ("Spine2","Leaf2","connected_to"),
    ("Leaf2","AppServer","gateway_for"),
]
for a,b,rel in edges:
    G.add_edge(a,b,relation=rel)

print("Path Client-VLAN120 -> AppServer:")
print(nx.shortest_path(G, "Client-VLAN120", "AppServer"))

## GraphRAG is not one algorithm

The term is used for several patterns:
- graph traversal before generation,
- entity/relation extraction into a graph,
- community summaries,
- combining vector retrieval with graph neighborhoods,
- querying an existing authoritative graph/CMDB.

For a NOC, the most defensible approach is often to use the **existing source-of-truth topology graph** rather than asking an LLM to invent one.

In [ ]:
node = "Leaf1"
neighbors = [(n, G.edges[node,n]["relation"]) for n in G.successors(node)]
print("Leaf1 outgoing relationships:", neighbors)

### Exercise

Add `Firewall1` between Leaf2 and AppServer, then ask:
- which endpoints depend on Firewall1?
- what is the blast radius if Leaf1 fails?
- which paths are redundant?

These are graph questions before they are language questions.